In [5]:
import pandas as pd
import numpy as np
import random
from transformers import pipeline

# 1. GENERATE MOCK CRM DUMP (UNSTRUCTURED DATA)
# Simulating realistic, messy lab/researcher feedback
interaction_texts = [
    "Does this microarray play nice with our current LIMS? not trying to rip and replace the whole system rn.",
    "data looks solid but $ per sample is way too high for our R01 grant budget. any volume discounts?",
    "Need the actual clinical validation papers. what are the accuracy metrics for the rare variants??",
    "worried about TAT. we have 500 samples backed up, can you actually hit the 2-day guarantee?",
    "if we commit to 10k units annually what's the pricing tier look like?",
    "techs are asking if the API docs are updated. they want to build custom pipelines and the old docs were useless."
]

np.random.seed(42)
n_records = 500

data = {
    "customer_id": [f"CUST-{i}" for i in range(1000, 1000 + n_records)],
    "raw_crm_notes": [random.choice(interaction_texts) for _ in range(n_records)],
    "campaign_received": np.random.choice(['Campaign A (ROI)', 'Campaign B (Tech Specs)'], n_records)
}

df = pd.DataFrame(data)

# Simulate historical conversion data
df['converted'] = np.random.choice([0, 1], n_records, p=[0.7, 0.3])


# 2. NLP EXTRACTION (Hugging Face)
print("Loading zero-shot model...")
# Using bart-large-mnli for quick zero-shot classification.
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

behavioral_segments = [
    "Price Sensitive",
    "Technical Integration",
    "Clinical Efficacy",
    "Turnaround Time"
]

def extract_behavior(text):
    result = classifier(text, candidate_labels=behavioral_segments)
    return result['labels'][0]

print("Processing unstructured text...")
# NOTE: Running row-by-row for the PoW. In production, this requires batched inference for speed.
# Taking head(10) just to speed up the Colab execution for the demo.
df_sample = df.head(10).copy()
df_sample['inferred_segment'] = df_sample['raw_crm_notes'].apply(extract_behavior)

# 3. COMMERCIAL TRANSLATION (NEXT BEST ACTION ENGINE)
# Map the ML classifications directly to a marketing/sales action
def get_nba(customer_segment):
    strategy_map = {
        "Price Sensitive": "Trigger automated discount sequence + ROI case study.",
        "Technical Integration": "Route to Sales Engineer + send updated API docs.",
        "Clinical Efficacy": "Send peer-reviewed whitepapers + invite to Clinical Webinar.",
        "Turnaround Time": "Highlight 2-day processing guarantee in next touchpoint."
    }
    return strategy_map.get(customer_segment, "Standard Nurture Sequence")

df_sample['next_best_action'] = df_sample['inferred_segment'].apply(get_nba)

print("\n--- PIPELINE OUTPUT: FROM RAW TEXT TO BUSINESS DECISION ---\n")
pd.set_option('display.max_colwidth', None)
print(df_sample[['raw_crm_notes', 'inferred_segment', 'next_best_action']])

Loading zero-shot model...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Processing unstructured text...

--- PIPELINE OUTPUT: FROM RAW TEXT TO BUSINESS DECISION ---

                                                                                                      raw_crm_notes  \
0          Does this microarray play nice with our current LIMS? not trying to rip and replace the whole system rn.   
1                       worried about TAT. we have 500 samples backed up, can you actually hit the 2-day guarantee?   
2  techs are asking if the API docs are updated. they want to build custom pipelines and the old docs were useless.   
3                 Need the actual clinical validation papers. what are the accuracy metrics for the rare variants??   
4                 data looks solid but $ per sample is way too high for our R01 grant budget. any volume discounts?   
5                 Need the actual clinical validation papers. what are the accuracy metrics for the rare variants??   
6                 data looks solid but $ per sample is way too high for o

In [ ]:
from google.colab import drive
drive.mount('/content/drive')